In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# 1. What are the mean and standard deviation of the annual US population across 2013-2018 inclusive?
population_df = spark.read.table("workspace.gold.population")
country_df = spark.read.table("workspace.gold.country")
answer = (
    population_df
    .join(
        country_df,
        population_df.nation_id == country_df.nation_id,
        "inner",
    )
    .filter(
        (F.col("year") >= 2013) &
        (F.col("year") <= 2018) &
        (F.col("nation") == "United States")
    )
    .agg(
        F.mean("population").alias("mean"),
        F.stddev("population").alias("stddev"),
    )
)
display(answer)

In [0]:
'''
2. For every series_id in the BLS data, what's the best year, meaning the year with the largest sum of value across all its quarters? (e.g. if PRS30006011 has values 1, 2 in 1995 and 3, 4 in 1996, its best year is 1996 with a summed value of 7.) Include enough of a human-readable label for each series that someone unfamiliar with BLS series codes could understand at a glance what's actually being measured.
'''
data_df = spark.read.table("workspace.gold.data")
class_df = spark.read.table("workspace.gold.class")
sector_df = spark.read.table("workspace.gold.sector")
seasonal_df = spark.read.table("workspace.gold.seasonal")
annual = (
    data_df
    .filter(F.col("quarter").isin("Q01", "Q02", "Q03", "Q04"))
    .groupBy("series_id", "year")
    .agg(
        F.sum("value").alias("annual_sum")
    )
)

window = Window.partitionBy("series_id").orderBy(
    F.col("annual_sum").desc()
)

answer = (
    annual
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)
answer = (
    answer
    .join(
        class_df,
        answer.nation_id == class_df.nation_id,
        "inner",
    )
)
display(answer)

In [0]:
# 3. For series_id = PRS30006032 and period = Q01, what was the value each year, joined with that year's population where available?
data_df = spark.table("workspace.gold.data")
population_df = spark.table("workspace.gold.population")

result = (
    data_df
    .filter(
        (F.col("series_id") == "PRS30006032") 
        # (F.col("quarter") == "Q01")
    )
    # .withColumn("year", F.col("year").cast("int"))
    # .join(
    #     population_df
    #     .withColumn("Year", F.col("Year").cast("int"))
    #     .select(
    #         F.col("Year").alias("population_year"),
    #         F.col("Population").alias("population")
    #     ),
    #     F.col("year") == F.col("population_year"),
    #     "left"
    # )
    # .select(
    #     "year",
    #     "quarter",
    #     "value",
    #     "population"
    # )
    # .orderBy("year")
)

display(result)